# CysMutML: leakage-aware ML for cysteine engineering

This portfolio notebook reconstructs the key conclusions from versioned artifacts. It does not require the private/raw FireProtDB export and does not retrain the production model.

**Question:** can simple physicochemical mutation descriptors provide a useful, interpretable stability baseline for X→Cys prioritization without leaking mutations from the same protein across folds?

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'results' / 'physchem_model_comparison'
metrics = pd.read_csv(RESULTS / 'regression_cv_metrics.csv')
cys_metrics = pd.read_csv(RESULTS / 'cys_specific_metrics.csv')
metrics.head()

## 1. Evaluation design

The primary split is `GroupKFold(protein_id)`. Mutations from one protein therefore remain in one fold. A random mutation-level split would answer an easier and potentially optimistic question.

In [ ]:
summary = metrics.groupby('model')[['mae', 'rmse', 'r2', 'pearson', 'spearman']].agg(['mean', 'std'])
summary.round(3)

In [ ]:
order = ['dummy_mean', 'ridge', 'hist_gradient_boosting']
means = metrics.groupby('model')['mae'].mean().reindex(order)
errors = metrics.groupby('model')['mae'].std().reindex(order)
ax = means.plot.bar(yerr=errors, capsize=4, color=['#9ca3af', '#2563eb', '#14b8a6'])
ax.set(title='Protein-grouped cross-validation', ylabel='MAE (kcal/mol)', xlabel='')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 2. X→Cys performance

The downstream use case is evaluated separately instead of assuming that aggregate performance transfers to cysteine substitutions.

In [ ]:
cys_summary = cys_metrics.groupby('model')[['mae', 'rmse', 'r2', 'pearson', 'spearman']].agg(['mean', 'std'])
cys_summary.round(3)

In [ ]:
overall = metrics.groupby('model')['mae'].mean().rename('all mutations')
cys = cys_metrics.groupby('model')['mae'].mean().rename('X→Cys')
pd.concat([overall, cys], axis=1).reindex(order).plot.bar(color=['#2563eb', '#f59e0b'])
plt.ylabel('MAE (kcal/mol)')
plt.xlabel('')
plt.title('Headline and task-specific error')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 3. Interpretability

Ridge is deployed despite a small HGB advantage because the coefficients expose the direction and relative contribution of the standardized descriptors.

In [ ]:
coef = pd.read_csv(ROOT / 'results' / 'model_interpretability' / 'ridge_coefficients.csv')
value_col = 'coefficient' if 'coefficient' in coef.columns else coef.select_dtypes('number').columns[-1]
feature_col = 'feature' if 'feature' in coef.columns else coef.columns[0]
top = coef.assign(abs_value=coef[value_col].abs()).nlargest(15, 'abs_value').sort_values(value_col)
top.plot.barh(x=feature_col, y=value_col, legend=False, color=['#ef4444' if x < 0 else '#2563eb' for x in top[value_col]])
plt.xlabel('Standardized Ridge coefficient')
plt.ylabel('')
plt.title('Largest linear effects')
plt.tight_layout()
plt.show()

## 4. From ML output to engineering ranking

The learned destabilization estimate is not the final answer. Target-PDB accessibility and other structural diagnostics are combined later through a transparent heuristic.

In [ ]:
ranking = pd.read_csv(ROOT / 'examples' / 'real_case' / 'residue_ranking.csv')
columns = ['rank_engineering', 'mutation', 'predicted_destabilization_ddg', 'relative_sasa', 'cys_site_suitability', 'rigidification_potential', 'final_engineering_score']
ranking[columns].head(10).round(3)

In [ ]:
top = ranking.nsmallest(15, 'rank_engineering').sort_values('final_engineering_score')
top.plot.barh(x='mutation', y='final_engineering_score', legend=False, color='#14b8a6')
plt.xlabel('Final engineering score (heuristic)')
plt.ylabel('')
plt.title('Top 15 candidates for PDB 1CSP, chain A')
plt.tight_layout()
plt.show()

## 5. Honest conclusion

- Physicochemical descriptors beat a mean baseline under protein-grouped validation.
- HGB is slightly stronger, while Ridge is easier to interpret and operationalize.
- X→Cys performance remains modest and is reported explicitly.
- The model predicts mutation-associated destabilization—not activity or immobilization success.
- The strongest next validation is grouping by sequence-identity clusters to reduce homolog leakage.